# LLM Evaluation on Greek Protipa Exams

In [3]:
import json
import logging
import requests
import os
import sys
import random
import time
import traceback
from pathlib import Path

import lm_eval
from lm_eval.utils import make_table
import pandas as pd
import yaml
from datasets import load_dataset
from datasets import load_dataset, concatenate_datasets
from dotenv import load_dotenv, find_dotenv
from lm_eval.models.openai_completions import OpenAIChatCompletion
from lm_eval.tasks import ConfigurableTask, TaskManager

import IPython.display

# Setup Logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

## 1. Environment Setup

In [6]:
load_dotenv(find_dotenv())

# API Config
os.environ["LITELLM_ILSP_EVAL_API_KEY"] = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
os.environ["LITELLM_HOST"] = os.getenv("LITELLM_HOST")
os.environ["OPENAI_API_KEY"] = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
api_base = os.getenv("LITELLM_HOST")

# Output Config
results_dir = Path(os.getenv("RESULTS_DIR", "tmp"))
results_dir.mkdir(parents=True, exist_ok=True)

models_to_test = ["gemma3-27b-it", "krikri-dpo-context"]
logger.info(f"Target models: {models_to_test}")

2026-05-06 16:13:12 - INFO - Target models: ['gemma3-27b-it', 'krikri-dpo-context']


In [ ]:
project_root = Path.cwd().parent 
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))
    logger.info(f"✅ Προστέθηκε το {src_path} στο path!")

try:
    from protipa_exams_dataset.data_loader import load_protipa_dataset, apply_matching_processing
    logger.info("🚀 Επιτυχία! Το data_loader φορτώθηκε από το src.")
except ImportError as e:
    logger.warning(f"⚠️ Δεν βρέθηκε η συνάρτηση/module. Έλεγξε τα ονόματα στο src. Error: {e}")

Διαθέσιμα μοντέλα

In [7]:
#api_key = os.getenv("OPENAI_API_KEY")
#api_base = os.getenv("OPENAI_BASE_URL")

api_key = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
api_base = os.getenv("LITELLM_HOST")

# Καθαρισμός URL
if api_base.endswith("/chat/completions"):
    api_base = api_base.replace("/chat/completions", "")
if not api_base.endswith("/v1"):
    api_base = api_base.rstrip("/") + "/v1"

try:
    response = requests.get(
        f"{api_base}/models", 
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=10
    )
    
    if response.status_code == 200:
        data = response.json()
        models = data.get('data', [])
        
        targets = ['llama', 'mistral', 'gemma', 'gpt']
        found_models = []
        
        for m in models:
            mid = m['id']
            if any(t in mid.lower() for t in targets):
                found_models.append(mid)
        
        print(json.dumps(found_models, indent=4))
        
    else:
        print(f"Error: {response.text}")

except Exception as e:
    print(f"Connection Error: {e}")

[
    "llama-krikri-8b-instruct-v1.5"
]


In [ ]:
#Saving results from closed tasks

file_path = "../results/krikri_test/llama-krikri-8b-instruct-v1.5/sample_results_closed.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/krikri_test/llama-krikri-8b-instruct-v1.5/krikri_closed_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

Table was successfully saved in results folder!
|                     Tasks                     |Version|   Filter   |n-shot|  Metric   |   |Value|   |Stderr|
|-----------------------------------------------|-------|------------|-----:|-----------|---|----:|---|-----:|
| - greek_protipa_exams_language_closed         |Yaml   |strict-match|     0|exact_match|↑  | 0.44|±  |0.0709|
| - greek_protipa_exams_maths_closed            |Yaml   |strict-match|     0|exact_match|↑  | 0.22|±  |0.0592|
| - greek_protipa_exams_religious_studies_closed|Yaml   |strict-match|     0|exact_match|↑  | 0.60|±  |0.0700|



In [3]:
#Saving results from closed tasks (new prompts)

file_path = "../results/krikri_test_new_prompts/llama-krikri-8b-instruct-v1.5/sample_results_closed.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/krikri_test_new_prompts/llama-krikri-8b-instruct-v1.5/krikri_closed_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

Table was successfully saved in results folder!
|                     Tasks                     |Version|   Filter   |n-shot|  Metric   |   |Value |   |Stderr|
|-----------------------------------------------|-------|------------|-----:|-----------|---|-----:|---|-----:|
| - greek_protipa_exams_language_closed         |Yaml   |strict-match|     0|exact_match|↑  |0.3333|±  |0.3333|
| - greek_protipa_exams_maths_closed            |Yaml   |strict-match|     0|exact_match|↑  |0.3333|±  |0.3333|
| - greek_protipa_exams_religious_studies_closed|Yaml   |strict-match|     0|exact_match|↑  |1.0000|±  |0.0000|



In [6]:
#Saving results from open tasks

file_path = "../results/krikri_test/llama-krikri-8b-instruct-v1.5/sample_results_open.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/krikri_test/llama-krikri-8b-instruct-v1.5/krikri_open_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

Table was successfully saved in results folder!
|               Tasks                |Version|Filter|n-shot|     Metric     |   | Value |   |Stderr |
|------------------------------------|-------|------|-----:|----------------|---|------:|---|------:|
|greek_protipa_exams_open_aggregate  |       |none  |      |bertscore_f1_max|↑  | 0.6534|±  | 0.0224|
|                                    |       |none  |      |bleu_max        |↑  | 7.8674|±  | 6.7851|
|                                    |       |none  |      |rouge1_max      |↑  |17.0830|±  | 9.5057|
|                                    |       |none  |      |rouge2_max      |↑  | 8.2592|±  | 5.8118|
|                                    |       |none  |      |rougeL_max      |↑  |16.6545|±  | 9.3446|
| - greek_protipa_exams_language_open|Yaml   |none  |     0|bertscore_f1_max|↑  | 0.5742|±  | 0.0001|
|                                    |       |none  |     0|bleu_max        |↑  | 0.0963|±  | 0.0963|
|                                 

In [4]:
#Saving results from open tasks (new prompts)

file_path = "../results/krikri_test_new_prompts/llama-krikri-8b-instruct-v1.5/sample_results_open.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/krikri_test_new_prompts/llama-krikri-8b-instruct-v1.5/krikri_open_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

Table was successfully saved in results folder!
|               Tasks                |Version|Filter|n-shot|     Metric     |   | Value |   |Stderr |
|------------------------------------|-------|------|-----:|----------------|---|------:|---|------:|
|greek_protipa_exams_open_aggregate  |       |none  |      |bertscore_f1_max|↑  | 0.7681|±  | 0.0305|
|                                    |       |none  |      |bleu_max        |↑  | 7.7682|±  | 5.2195|
|                                    |       |none  |      |rouge1_max      |↑  |47.6321|±  | 8.0369|
|                                    |       |none  |      |rouge2_max      |↑  |12.1134|±  | 7.3912|
|                                    |       |none  |      |rougeL_max      |↑  |46.7031|±  | 8.3058|
| - greek_protipa_exams_language_open|Yaml   |none  |     0|bertscore_f1_max|↑  | 0.8843|±  | 0.0595|
|                                    |       |none  |     0|bleu_max        |↑  | 3.5607|±  | 3.5607|
|                                 

In [ ]:
#Saving results from open tasks (new prompts_2)
#Best bertscore for open tasks so far

file_path = "../results/krikri_test_new_prompts_2/llama-krikri-8b-instruct-v1.5/sample_results_open.json"

with open (file_path, "r", encoding="utf-8") as f:
    data=json.load(f)

results = data["results"]

df = pd.DataFrame(results).T

df.to_excel("../results/krikri_test_new_prompts_2/llama-krikri-8b-instruct-v1.5/krikri_open_results.xlsx")

print ("Table was successfully saved in results folder!")
print(make_table(data))

Table was successfully saved in results folder!
|               Tasks                |Version|Filter|n-shot|     Metric     |   | Value |   |Stderr |
|------------------------------------|-------|------|-----:|----------------|---|------:|---|------:|
|greek_protipa_exams_open_aggregate  |       |none  |      |bertscore_f1_max|↑  | 0.7572|±  | 0.0233|
|                                    |       |none  |      |bleu_max        |↑  | 4.6300|±  | 3.0027|
|                                    |       |none  |      |rouge1_max      |↑  |34.2100|±  | 9.3787|
|                                    |       |none  |      |rouge2_max      |↑  |11.6951|±  | 7.3202|
|                                    |       |none  |      |rougeL_max      |↑  |32.7217|±  | 9.3880|
| - greek_protipa_exams_language_open|Yaml   |none  |     0|bertscore_f1_max|↑  | 0.9025|±  | 0.0599|
|                                    |       |none  |     0|bleu_max        |↑  | 0.0000|±  | 0.0000|
|                                 